In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [7]:
train = pd.read_csv('clean_data/clean_data_train.csv')
train.head()

,time,wind_speed_10m,temperature_2m,cloud_cover,relative_humidity_2m,surface_pressure,apparent_temperature,precipitation,vapour_pressure_deficit
0,2021-01-01 00:00:00,26.144030,19.50,100.0,70.98951,1020.28920,17.023354,0.0,0.657872
1,2021-01-01 01:00:00,24.192429,19.50,100.0,68.94631,1019.78986,17.147188,0.0,0.704225
2,2021-01-01 02:00:00,23.565569,19.35,99.0,67.36236,1019.09000,16.914827,0.0,0.733305
3,2021-01-01 03:00:00,24.192429,19.20,91.0,68.21654,1018.59010,16.690643,0.0,0.707482
4,2021-01-01 04:00:00,23.838959,19.25,100.0,66.90323,1018.89010,16.706848,0.0,0.739021


# 1. EDA

## 1.1 Kiểm tra null, ouliers

In [8]:
print("missing_data:", train.isna().sum())
print("unique dates:", train.index.nunique())
print("len train: ", len(train))

missing_data: time                       0
wind_speed_10m             0
temperature_2m             0
cloud_cover                0
relative_humidity_2m       0
surface_pressure           0
apparent_temperature       0
precipitation              0
vapour_pressure_deficit    0
dtype: int64
unique dates: 42573
len train:  42573


Nhờ việc thiết lập tần suất (asfreq) ở bước làm sạch, việc kiểm tra khuyết thiếu bằng isna() cho thấy dữ liệu hoàn toàn liên tục

## Kiểm tra trend, tính mùa vụ, chu kỳ

## Kiểm tra phân phối của dữ liệu

## Kiểm tra tính dừng của dữ liệu

## Kiểm tra tính tự tương quan

## Kiểm tra biến ngoại sinh

# 2. Feature Engineering

# 3. Training

In [ ]:
print("="*50)
print("BƯỚC 4: TÁCH BIẾN NGOẠI SINH (FEATURES) VÀ BIẾN MỤC TIÊU (TARGET)")
print("="*50)

# Định nghĩa danh sách các đặc trưng (bao gồm biến trễ và biến ngoại sinh)
# Nếu dữ liệu của bạn có thêm cột 'relative_humidity_2m (%)' hoặc 'rain', hãy thêm vào list này nhé!
feature_cols = [
    'hour', 'day_of_week', 'day_of_month', 'month', 'day_of_year',
    'lag_1', 'lag_2', 'lag_3', 'lag_24',
    'rolling_mean_24', 'rolling_std_24'
]

target_col = 'temperature_2m (°C)'

# Tách thành ma trận X và mảng y
X_train = train_df[feature_cols]
y_train = train_df[target_col]

print(f"✅ Đã cấu hình xong không gian đặc trưng (Feature Space):")
print(f" - Số lượng biến ngoại sinh và biến trễ đầu vào (X): {len(feature_cols)} biến.")
print(f" - Biến mục tiêu cần dự báo (y): '{target_col}'")
print(f" - Kích thước X_train: {X_train.shape}")
print(f" - Kích thước y_train: {y_train.shape}")

In [ ]:
import xgboost as xgb
import pickle
import os

print("="*50)
print("BƯỚC 5: HUẤN LUYỆN MÔ HÌNH XGBOOST REGRESSOR")
print("="*50)

# 1. KHỞI TẠO MÔ HÌNH XGBOOST
# Cấu hình các tham số để kiểm soát overfitting và tối ưu cho chuỗi thời gian
model = xgb.XGBRegressor(
    n_estimators=100,      # Số lượng cây quyết định
    max_depth=6,           # Độ sâu tối đa của cây
    learning_rate=0.1,     # Tốc độ học
    subsample=0.8,         # Tỷ lệ lấy mẫu dòng cho mỗi cây (chống overfitting)
    colsample_bytree=0.8,  # Tỷ lệ lấy mẫu cột (biến ngoại sinh) cho mỗi cây
    random_state=42,
    n_jobs=-1              # Sử dụng tối đa hiệu năng CPU để train nhanh
)

# 2. TIẾN HÀNH HUẤN LUYỆN (FIT MODEL)
print("-> Đang huấn luyện mô hình XGBoost với các biến ngoại sinh...")
model.fit(X_train, y_train)
print("✅ Huấn luyện mô hình thành công!")

# 3. ĐÓNG GÓI VÀ LƯU TRỮ MÔ HÌNH (ĐỂ DÙNG BÊN FILE TESTING.IPYNB)
# Tạo thư mục lưu trữ nếu chưa có
model_dir = "saved_models"
os.makedirs(model_dir, exist_ok=True)

# Lưu model XGBoost
model_path = os.path.join(model_dir, "xgboost_weather_model.pkl")
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# Lưu kèm danh sách cột thuộc tính để bên test_df trích xuất khớp 100%
features_path = os.path.join(model_dir, "feature_columns.pkl")
with open(features_path, 'wb') as f:
    pickle.dump(feature_cols, f)

print(f"\n📦 Đã đóng gói và lưu trữ thành công:")
print(f" - File mô hình: {model_path}")
print(f" - File danh sách đặc trưng: {features_path}")